In [5]:
import pandas as pd
from pathlib import Path
import os


os.chdir('C:\\Users\\User\\Documents\\Coding_projects\\FPL\\FPL_points-prediction')
print(f"Current working directory: {os.getcwd()}")

INGESTION_DIR_NAME = "ingested"

DATA_ROOT = Path("data")
INGESTION_DIR_PATH = DATA_ROOT / INGESTION_DIR_NAME

Current working directory: C:\Users\User\Documents\Coding_projects\FPL\FPL_points-prediction


In [6]:
df_24_25 = pd.read_csv(INGESTION_DIR_PATH / "fpl-core-insights" / "2024-2025" / "players.csv")
df_25_26 = pd.read_csv(INGESTION_DIR_PATH / "fpl-core-insights" / "2025-2026" / "players.csv")

# Comparing 2024-2025 and 2025-2026 seasons

In [31]:
set(df_25_26.columns) - set(df_24_25.columns)

{'away_distance_covered',
 'away_number_of_sprints',
 'away_running_distance',
 'away_sprinting_distance',
 'away_top_speed',
 'away_walking_distance',
 'home_distance_covered',
 'home_number_of_sprints',
 'home_running_distance',
 'home_sprinting_distance',
 'home_top_speed',
 'home_walking_distance',
 'tournament'}

In [32]:
set(df_24_25.columns).intersection(set(df_25_26.columns))

{'away_accurate_crosses',
 'away_accurate_crosses_pct',
 'away_accurate_long_balls',
 'away_accurate_long_balls_pct',
 'away_accurate_passes',
 'away_accurate_passes_pct',
 'away_aerial_duels_won',
 'away_aerial_duels_won_pct',
 'away_big_chances',
 'away_big_chances_missed',
 'away_blocked_shots',
 'away_blocks',
 'away_clearances',
 'away_corners',
 'away_duels_won',
 'away_expected_goals_xg',
 'away_fouls_committed',
 'away_ground_duels_won',
 'away_ground_duels_won_pct',
 'away_hit_woodwork',
 'away_interceptions',
 'away_keeper_saves',
 'away_non_penalty_xg',
 'away_offsides',
 'away_opposition_half',
 'away_own_half',
 'away_passes',
 'away_possession',
 'away_red_cards',
 'away_score',
 'away_shots_inside_box',
 'away_shots_off_target',
 'away_shots_on_target',
 'away_shots_outside_box',
 'away_successful_dribbles',
 'away_successful_dribbles_pct',
 'away_tackles_won',
 'away_tackles_won_pct',
 'away_team',
 'away_team_elo',
 'away_throws',
 'away_total_shots',
 'away_touches_in

In [28]:
df_25_26

,gameweek,kickoff_time,home_team,home_team_elo,home_score,away_score,away_team,away_team_elo,finished,match_id,...,away_walking_distance,home_running_distance,away_running_distance,home_sprinting_distance,away_sprinting_distance,home_number_of_sprints,away_number_of_sprints,home_top_speed,away_top_speed,tournament
0,1.0,2024-08-17 14:00:00,3.0,1946.90,2.0,0.0,39.0,1677.86,True,24-25-prem-arsenal-vs-wolverhampton-wanderers,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1.0,2024-08-18 13:00:00,94.0,1711.08,2.0,1.0,31.0,1759.71,True,24-25-prem-brentford-vs-crystal-palace,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1.0,2024-08-18 15:30:00,8.0,1810.12,0.0,2.0,43.0,2050.57,True,24-25-prem-chelsea-vs-manchester-city,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1.0,2024-08-17 14:00:00,11.0,1706.85,0.0,3.0,36.0,1713.16,True,24-25-prem-everton-vs-brighton-&-hove-albion,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1.0,2024-08-17 11:30:00,40.0,1568.33,0.0,2.0,14.0,1900.69,True,24-25-prem-ipswich-town-vs-liverpool,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
900,38.0,NaN,6.0,1777.31,1.0,0.0,11.0,1803.43,True,25-26-prem-tottenham-hotspur-vs-everton,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,prem
901,38.0,NaN,56.0,1735.93,2.0,1.0,8.0,1831.10,True,25-26-prem-sunderland-vs-chelsea,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,prem
902,38.0,NaN,31.0,1803.82,1.0,2.0,3.0,2063.76,True,25-26-prem-crystal-palace-vs-arsenal,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,prem
903,38.0,NaN,90.0,1666.34,1.0,1.0,39.0,1679.69,True,25-26-prem-burnley-vs-wolverhampton-wanderers,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,prem


# Verifying Name Code stability

Looks pretty good. Just name formatting changes between seasons. I will solve by accepting the most recent name representations

In [21]:
import pandas as pd
import unicodedata
import re


def merge_seasons(df_a, df_b, season_a='2024-2025', season_b='2025-2026', on='player_code'):
    """Outer-merge two season dataframes on a key, suffixing columns by season."""
    return pd.merge(df_a, df_b, how='outer', on=on, suffixes=(f'_{season_a}', f'_{season_b}'))


def get_season_cols(df, season):
    """Return all columns belonging to a given season suffix."""
    suffix = f'_{season}'
    return [col for col in df.columns if col.endswith(suffix)]


def check_var_equal(df, var_name, seasons=('2024-2025', '2025-2026')):
    """Boolean mask: True where var_name matches across both seasons (NaN != NaN -> False)."""
    col_a, col_b = (f'{var_name}_{season}' for season in seasons)
    return df[col_a] == df[col_b]


def get_missing_masks(df, seasons=('2024-2025', '2025-2026')):
    """Boolean masks for rows entirely missing from each season (i.e. player not present)."""
    return {season: df[get_season_cols(df, season)].isna().all(axis=1) for season in seasons}


def normalize_name(name):
    """Strip diacritics, lowercase, collapse whitespace. Note: doesn't catch distinct-letter
    swaps like ø/ö, or first/second name order swaps — those need manual review either way."""
    if pd.isna(name):
        return name
    nfkd = unicodedata.normalize('NFKD', name)
    stripped = ''.join(c for c in nfkd if not unicodedata.combining(c))
    return re.sub(r'\s+', ' ', stripped).strip().lower()


def split_player_matches(df, seasons=('2024-2025', '2025-2026')):
    """
    Given a merged season df, return:
      - df_only_in_one_season: players present in only one season
      - df_matched: players present in both seasons (same player_code)
      - df_name_mismatches: matched players whose names differ (raw comparison)
      - df_name_mismatches_real: subset of the above that still differ after normalization
        (i.e. not just diacritics/nickname/surname-length noise — worth a manual look)
    """
    missing = get_missing_masks(df, seasons)
    missing_any = missing[seasons[0]] | missing[seasons[1]]

    df_only_in_one_season = df[missing_any]
    df_matched = df[~missing_any]

    names_equal_raw = check_var_equal(df_matched, 'first_name', seasons) & check_var_equal(df_matched, 'second_name', seasons)
    df_name_mismatches = df_matched[~names_equal_raw]

    normalized_first = pd.concat(
        [df_name_mismatches[f'first_name_{s}'].apply(normalize_name) for s in seasons], axis=1
    )
    normalized_second = pd.concat(
        [df_name_mismatches[f'second_name_{s}'].apply(normalize_name) for s in seasons], axis=1
    )
    names_equal_normalized = (normalized_first.iloc[:, 0] == normalized_first.iloc[:, 1]) & \
                              (normalized_second.iloc[:, 0] == normalized_second.iloc[:, 1])

    df_name_mismatches_real = df_name_mismatches[~names_equal_normalized]

    return df_only_in_one_season, df_matched, df_name_mismatches, df_name_mismatches_real


def check_orphan_code_churn(df_only_in_one_season, seasons=('2024-2025', '2025-2026')):
    """
    For players only present in one season, check whether a normalized second_name
    match exists in the *other* season's orphans — looser than full-name matching,
    since first/second name token counts are inconsistent even for confirmed-same players.
    """
    missing = get_missing_masks(df_only_in_one_season, seasons)
    orphans_a = df_only_in_one_season[~missing[seasons[0]]]
    orphans_b = df_only_in_one_season[~missing[seasons[1]]]

    surnames_a = orphans_a[f'second_name_{seasons[0]}'].apply(normalize_name)
    surnames_b = orphans_b[f'second_name_{seasons[1]}'].apply(normalize_name)

    matched_surnames = set(surnames_a.dropna()) & set(surnames_b.dropna())

    suspected_code_churn = pd.concat([
        orphans_a[surnames_a.isin(matched_surnames)],
        orphans_b[surnames_b.isin(matched_surnames)],
    ])

    return suspected_code_churn.sort_values(
        by=[f'second_name_{seasons[0]}', f'second_name_{seasons[1]}']
    )

def summarise_player_code_integrity(df_a, df_b, seasons=('2024-2025', '2025-2026'), on='player_code'):
    """
    Full verification pass on player_code as a join key across two seasons.
    Prints a summary and returns all intermediate frames for inspection.
    """
    df_merged = merge_seasons(df_a, df_b, seasons[0], seasons[1], on=on)

    df_only_in_one_season, df_matched, df_name_mismatches, df_name_mismatches_real = \
        split_player_matches(df_merged, seasons)

    suspected_code_churn = check_orphan_code_churn(df_only_in_one_season, seasons)

    print(f"Total merged rows:                {len(df_merged)}")
    print(f"Matched on {on} (both seasons):   {len(df_matched)}")
    print(f"  - raw name mismatches:          {len(df_name_mismatches)}")
    print(f"  - unresolved after normalizing: {len(df_name_mismatches_real)}  <- worth a manual look")
    print(f"Only in one season:                {len(df_only_in_one_season)}")
    print(f"  - suspected code churn (name match in other season): {len(suspected_code_churn)}  <- worth a manual look")

    return {
        'df_merged': df_merged,
        'df_matched': df_matched,
        'df_name_mismatches': df_name_mismatches,
        'df_name_mismatches_real': df_name_mismatches_real,
        'df_only_in_one_season': df_only_in_one_season,
        'suspected_code_churn': suspected_code_churn,
    }


# --- Usage ---
results = summarise_player_code_integrity(df_24_25, df_25_26)

results['df_name_mismatches_real'][
    ['first_name_2024-2025', 'second_name_2024-2025', 'first_name_2025-2026', 'second_name_2025-2026']
]

results['suspected_code_churn'][
    ['player_code', 'first_name_2024-2025', 'second_name_2024-2025', 'first_name_2025-2026', 'second_name_2025-2026']
]

Total merged rows:                1111
Matched on player_code (both seasons):   534
  - raw name mismatches:          55
  - unresolved after normalizing: 47  <- worth a manual look
Only in one season:                577
  - suspected code churn (name match in other season): 21  <- worth a manual look


,player_code,first_name_2024-2025,second_name_2024-2025,first_name_2025-2026,second_name_2025-2026
710,494041,Ethan,Brierley,NaN,NaN
20,55459,Aaron,Cresswell,NaN,NaN
750,500926,Ronnie,Edwards,NaN,NaN
823,524180,Cameron,Humphreys,NaN,NaN
353,222018,Ben,Johnson,NaN,NaN
987,592736,Maeson,King,NaN,NaN
713,494551,Jayden,Moore,NaN,NaN
276,204727,Malang,Sarr,NaN,NaN
754,502222,Ramón,Sosa,NaN,NaN
800,516250,Jacob,Wright,NaN,NaN
